# 面试题：怎样从零实现带图结构约束的 Graph Transformer？

## 可以直接复述的回答

Graph Transformer 仍用 query、key、value 计算注意力，但必须把图结构编码进可见性或 attention bias，否则它会退化为把所有节点视为全连接序列。最简单的实现是给每个节点只开放自身和一跳邻居，堆两层后信息最多传播两跳。mask 要在 softmax 前把禁止位置设为极小值，使其概率严格为零。残差连接和逐节点归一化帮助训练稳定，输出可用于节点分类或图级任务。本例标签定义为“是否在已知欺诈种子两跳内”，只看自身特征的基线无法识别邻居风险。训练时整张图的特征与边都参与消息传播、只有测试标签被隐藏，因此这是 transductive 节点分类，不是对新图的 inductive 泛化。评估预先固定为两类 logits 的 argmax，也就是风险概率 0.5，不根据测试结果事后调阈值。最后把 mask 改成全连接，复现不相干分量也能直接关注欺诈种子的结构泄漏。

## 真实案例

十个脱敏账户构成两个不连通设备共享分量，A0 是已确认欺诈种子，A1/A2 为两跳内高风险，其他账户正常。7 个节点标签用于训练、3 个节点标签只用于测试；但十个节点的特征和图边在训练 forward 中全部可见，所以结论严格限定为 transductive。金额和账户年龄是简化特征，不能用于真实风控。

In [1]:
from pprint import pprint  # 导入结构化打印函数以展示账户、注意力和结果
import math  # 导入平方根函数以缩放点积注意力
import torch  # 导入 PyTorch 以实现真实 Graph Transformer 训练
torch.manual_seed(47)  # 固定参数初始化保证结果可复现
torch.set_num_threads(1)  # 限制 CPU 线程以稳定小图实验
nodes = [{"id": "A0", "seed": 1.0, "amount": 0.7, "age": 0.4, "label": 1, "split": "train"}, {"id": "A1", "seed": 0.0, "amount": 0.5, "age": 0.5, "label": 1, "split": "train"}, {"id": "A2", "seed": 0.0, "amount": 0.5, "age": 0.5, "label": 1, "split": "test"}, {"id": "A3", "seed": 0.0, "amount": 0.5, "age": 0.5, "label": 0, "split": "train"}, {"id": "A4", "seed": 0.0, "amount": 0.5, "age": 0.5, "label": 0, "split": "train"}, {"id": "B0", "seed": 0.0, "amount": 0.5, "age": 0.5, "label": 0, "split": "train"}, {"id": "B1", "seed": 0.0, "amount": 0.5, "age": 0.5, "label": 0, "split": "train"}, {"id": "B2", "seed": 0.0, "amount": 0.5, "age": 0.5, "label": 0, "split": "train"}, {"id": "B3", "seed": 0.0, "amount": 0.5, "age": 0.5, "label": 0, "split": "test"}, {"id": "B4", "seed": 0.0, "amount": 0.5, "age": 0.5, "label": 0, "split": "test"}]  # 构造两个不连通账户分量和三枚测试节点
edges = [("A0", "A1"), ("A1", "A2"), ("A2", "A3"), ("A3", "A4"), ("B0", "B1"), ("B1", "B2"), ("B2", "B3"), ("B3", "B4")]  # 构造两条设备共享链路
node_to_index = {node["id"]: index for index, node in enumerate(nodes)}  # 建立账户 ID 到矩阵行号的映射
features = torch.tensor([[node["seed"], node["amount"], node["age"]] for node in nodes], dtype=torch.float32)  # 构造十乘三节点特征矩阵
targets = torch.tensor([node["label"] for node in nodes], dtype=torch.long)  # 构造节点风险标签张量
train_mask = torch.tensor([node["split"] == "train" for node in nodes], dtype=torch.bool)  # 构造七个训练节点掩码
test_mask = ~train_mask  # 构造三个测试节点掩码
adjacency = torch.zeros((len(nodes), len(nodes)), dtype=torch.bool)  # 创建无向图连接矩阵
for left, right in edges:  # 遍历八条共享设备边
    adjacency[node_to_index[left], node_to_index[right]] = True  # 写入正向连接
    adjacency[node_to_index[right], node_to_index[left]] = True  # 写入反向连接
graph_mask = adjacency | torch.eye(len(nodes), dtype=torch.bool)  # 只允许自身与一跳邻居互相注意
print("账户图输入预览：")  # 输出真实案例标题
pprint(nodes)  # 展示十个节点的特征、标签和切分
print("设备共享边：", edges)  # 展示两个不连通分量的图结构

账户图输入预览：
[{'age': 0.4,
  'amount': 0.7,
  'id': 'A0',
  'label': 1,
  'seed': 1.0,
  'split': 'train'},
 {'age': 0.5,
  'amount': 0.5,
  'id': 'A1',
  'label': 1,
  'seed': 0.0,
  'split': 'train'},
 {'age': 0.5,
  'amount': 0.5,
  'id': 'A2',
  'label': 1,
  'seed': 0.0,
  'split': 'test'},
 {'age': 0.5,
  'amount': 0.5,
  'id': 'A3',
  'label': 0,
  'seed': 0.0,
  'split': 'train'},
 {'age': 0.5,
  'amount': 0.5,
  'id': 'A4',
  'label': 0,
  'seed': 0.0,
  'split': 'train'},
 {'age': 0.5,
  'amount': 0.5,
  'id': 'B0',
  'label': 0,
  'seed': 0.0,
  'split': 'train'},
 {'age': 0.5,
  'amount': 0.5,
  'id': 'B1',
  'label': 0,
  'seed': 0.0,
  'split': 'train'},
 {'age': 0.5,
  'amount': 0.5,
  'id': 'B2',
  'label': 0,
  'seed': 0.0,
  'split': 'train'},
 {'age': 0.5,
  'amount': 0.5,
  'id': 'B3',
  'label': 0,
  'seed': 0.0,
  'split': 'test'},
 {'age': 0.5,
  'amount': 0.5,
  'id': 'B4',
  'label': 0,
  'seed': 0.0,
  'split': 'test'}]
设备共享边： [('A0', 'A1'), ('A1', 'A2'), ('A2', '

## Baseline / 基线：只看自身 seed 标记

基线只把 seed=1 的节点判为高风险。测试节点都不是已确认种子，因此 A2 被漏掉，两个 B 分量节点正确判为正常，测试准确率为 2/3。

In [2]:
baseline_predictions = (features[:, 0] > 0.5).to(torch.long)  # 用自身欺诈种子标记生成基线预测
baseline_test_accuracy = float((baseline_predictions[test_mask] == targets[test_mask]).to(torch.float32).mean())  # 计算三个测试节点准确率
baseline_rows = []  # 创建逐测试节点基线结果表
for index, node in enumerate(nodes):  # 遍历十个账户节点
    if node["split"] == "test":  # 只记录三个测试节点
        baseline_rows.append({"节点": node["id"], "seed": node["seed"], "真实": node["label"], "预测": int(baseline_predictions[index])})  # 保存自身证据与预测
print("Seed-only Baseline：")  # 输出基线结果标题
pprint(baseline_rows)  # 展示 A2 的两跳风险无法由自身特征发现
print(f"Baseline test accuracy={baseline_test_accuracy:.3f}")  # 输出基线聚合指标

Seed-only Baseline：
[{'seed': 0.0, '真实': 1, '节点': 'A2', '预测': 0},
 {'seed': 0.0, '真实': 0, '节点': 'B3', '预测': 0},
 {'seed': 0.0, '真实': 0, '节点': 'B4', '预测': 0}]
Baseline test accuracy=0.667


## 手写 masked self-attention 与两层 Graph Transformer

每层显式计算 QKᵀ/√d，禁止边在 softmax 前填入 -1e9。随后用 attention@V 聚合、线性输出、残差和手写 LayerNorm；两层恰好允许 A0 信号传播到 A2。

In [3]:
def layer_norm(features, epsilon=1e-5):  # 定义逐节点手写 LayerNorm
    mean = features.mean(dim=1, keepdim=True)  # 计算每个节点隐藏维均值
    variance = ((features - mean) ** 2).mean(dim=1, keepdim=True)  # 计算每个节点隐藏维方差
    return (features - mean) / torch.sqrt(variance + epsilon)  # 返回标准化节点表示
class MaskedGraphAttention(torch.nn.Module):  # 定义单层图结构约束注意力
    def __init__(self, hidden_size):  # 初始化 Q、K、V 和输出参数
        super().__init__()  # 初始化 PyTorch 模块基类
        self.query_weight = torch.nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.2)  # 创建 query 映射
        self.key_weight = torch.nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.2)  # 创建 key 映射
        self.value_weight = torch.nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.2)  # 创建 value 映射
        self.output_weight = torch.nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.2)  # 创建注意力输出映射
    def forward(self, hidden, allowed_mask):  # 定义带图 mask 的注意力前向
        queries = hidden @ self.query_weight  # 计算所有节点 query
        keys = hidden @ self.key_weight  # 计算所有节点 key
        values = hidden @ self.value_weight  # 计算所有节点 value
        scores = queries @ keys.T / math.sqrt(hidden.shape[1])  # 计算节点两两缩放点积
        masked_scores = scores.masked_fill(~allowed_mask, -1e9)  # 在 softmax 前屏蔽非邻居节点
        shifted = masked_scores - masked_scores.max(dim=1, keepdim=True).values  # 减去行最大值保证数值稳定
        attention = shifted.exp() / shifted.exp().sum(dim=1, keepdim=True)  # 手写逐节点注意力概率
        messages = attention @ values  # 按注意力聚合允许邻居的 value
        output = layer_norm(hidden + messages @ self.output_weight)  # 执行输出映射、残差和归一化
        return torch.relu(output), attention, scores  # 返回节点表示、概率和未屏蔽分数
class TinyGraphTransformer(torch.nn.Module):  # 定义两层节点分类 Graph Transformer
    def __init__(self, input_size=3, hidden_size=12):  # 初始化输入投影、两层 attention 与分类头
        super().__init__()  # 初始化 PyTorch 模块基类
        self.input_weight = torch.nn.Parameter(torch.randn(input_size, hidden_size) * 0.25)  # 创建节点特征输入映射
        self.input_bias = torch.nn.Parameter(torch.zeros(hidden_size))  # 创建输入投影偏置
        self.layer1 = MaskedGraphAttention(hidden_size)  # 创建第一跳图注意力层
        self.layer2 = MaskedGraphAttention(hidden_size)  # 创建第二跳图注意力层
        self.classifier_weight = torch.nn.Parameter(torch.randn(hidden_size, 2) * 0.2)  # 创建节点风险分类权重
        self.classifier_bias = torch.nn.Parameter(torch.zeros(2))  # 创建节点分类偏置
    def forward(self, node_features, allowed_mask):  # 定义两跳图注意力分类前向
        hidden0 = torch.relu(node_features @ self.input_weight + self.input_bias)  # 把原始节点特征映射到隐藏空间
        hidden1, attention1, raw_scores1 = self.layer1(hidden0, allowed_mask)  # 执行第一跳结构注意力
        hidden2, attention2, raw_scores2 = self.layer2(hidden1, allowed_mask)  # 执行第二跳结构注意力
        logits = hidden2 @ self.classifier_weight + self.classifier_bias  # 输出每个节点两类 logits
        return logits, [hidden0, hidden1, hidden2], [attention1, attention2], [raw_scores1, raw_scores2]  # 返回分类和两层过程
model = TinyGraphTransformer()  # 实例化手写 Graph Transformer
with torch.no_grad():  # 关闭结构检查阶段梯度记录
    initial_logits, initial_hiddens, initial_attentions, initial_scores = model(features, graph_mask)  # 运行未训练图前向
print("Graph Transformer 张量 shape：", {"logits": tuple(initial_logits.shape), "hidden": [tuple(value.shape) for value in initial_hiddens], "attention": [tuple(value.shape) for value in initial_attentions]})  # 展示节点级张量形状
print("A2 第一层允许注意的节点：", [nodes[index]["id"] for index, allowed in enumerate(graph_mask[node_to_index["A2"]]) if bool(allowed)])  # 展示图 mask 的实际可见范围

Graph Transformer 张量 shape： {'logits': (10, 2), 'hidden': [(10, 12), (10, 12), (10, 12)], 'attention': [(10, 10), (10, 10)]}
A2 第一层允许注意的节点： ['A1', 'A2', 'A3']


## Transductive 节点监督与真实 backward

loss 只在 7 个训练节点标签上计算，A2/B3/B4 标签不参与更新；不过它们的特征和边仍在整图 forward 中参与消息传递。这正是 transductive node classification，不能表述成“模型泛化到新节点或新图”。图 mask 固定由业务边生成，梯度流经 Q、K、V、两层消息和分类头。

In [4]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.012)  # 使用基础 Adam 更新手写图注意力参数
training_ledger = []  # 创建 loss、训练准确率和 query 梯度账本
for epoch in range(601):  # 执行六百零一次全图节点训练
    optimizer.zero_grad()  # 清空上一轮参数梯度
    logits, hidden_states, attentions, raw_scores = model(features, graph_mask)  # 运行真实两层 Graph Transformer forward
    train_logits = logits[train_mask]  # 只选择七个训练节点 logits
    train_targets = targets[train_mask]  # 只选择七个训练节点标签
    log_probabilities = train_logits - torch.logsumexp(train_logits, dim=1, keepdim=True)  # 手写两类对数 softmax
    loss = -log_probabilities[torch.arange(train_targets.shape[0]), train_targets].mean()  # 计算训练节点平均交叉熵
    loss.backward()  # 运行真实 backward 计算 QKV 梯度
    query_gradient = float(model.layer1.query_weight.grad.norm())  # 读取第一层 query 权重梯度范数
    train_accuracy = float((train_logits.argmax(dim=1) == train_targets).to(torch.float32).mean())  # 计算训练节点准确率
    if epoch % 120 == 0:  # 每一百二十轮记录一次状态
        training_ledger.append({"epoch": epoch, "loss": round(float(loss), 6), "train_accuracy": round(train_accuracy, 3), "query_grad": round(query_gradient, 6)})  # 保存损失、准确率与真实梯度
    optimizer.step()  # 根据当前梯度更新全部手写参数
print("Graph Transformer 训练账本：")  # 输出训练过程标题
pprint(training_ledger)  # 展示 loss 与 query 梯度随训练变化

Graph Transformer 训练账本：
[{'epoch': 0,
  'loss': 0.580178,
  'query_grad': 0.001279,
  'train_accuracy': 0.714},
 {'epoch': 120, 'loss': 0.001112, 'query_grad': 6e-06, 'train_accuracy': 1.0},
 {'epoch': 240, 'loss': 0.000415, 'query_grad': 1e-06, 'train_accuracy': 1.0},
 {'epoch': 360, 'loss': 0.000228, 'query_grad': 1e-06, 'train_accuracy': 1.0},
 {'epoch': 480, 'loss': 0.000147, 'query_grad': 0.0, 'train_accuracy': 1.0},
 {'epoch': 600, 'loss': 0.000104, 'query_grad': 0.0, 'train_accuracy': 1.0}]


## 逐节点结果、两层注意力与结果解读

A2 第一层只能看到 A1/A2/A3，第二层的 A1 表示已经融合 A0，因此 A2 的风险排序可能上升。决策规则在看测试结果前固定为标准 argmax（等价于风险概率至少 0.5），不再使用事后挑选的 0.30。若 A2 概率高于 B 分量却仍低于 0.5，就应诚实报告“排序学到了、默认决策未改善”，而不能用测试标签反向迁就阈值。

In [5]:
decision_threshold = 0.5  # 在评估前固定标准二分类概率阈值。
with torch.no_grad():  # 关闭评估阶段梯度记录。
    final_logits, final_hiddens, final_attentions, final_scores = model(features, graph_mask)  # 运行训练完成后的 transductive 图前向。
    shifted = final_logits - final_logits.max(dim=1, keepdim=True).values  # 稳定节点 softmax 输入。
    probabilities = shifted.exp() / shifted.exp().sum(dim=1, keepdim=True)  # 手写节点风险概率。
    predictions = final_logits.argmax(dim=1)  # 使用预先固定的 argmax 而非测试集调阈值。
graph_transformer_accuracy = float((predictions[test_mask] == targets[test_mask]).to(torch.float32).mean())  # 计算三个隐藏标签节点的 accuracy。
result_rows = []  # 创建逐测试节点结果表。
for index, node in enumerate(nodes):  # 遍历十个账户节点。
    if node["split"] == "test":  # 只记录三个测试标签节点。
        result_rows.append({"节点": node["id"], "真实": node["label"], "Baseline": int(baseline_predictions[index]), "GraphTransformer@argmax": int(predictions[index]), "风险概率": round(float(probabilities[index, 1]), 4)})  # 保存同口径默认决策和概率。
a2_index = node_to_index["A2"]  # 读取两跳风险节点行号。
b_component_test_indices = [node_to_index["B3"], node_to_index["B4"]]  # 读取两个不连通测试节点行号。
a2_risk_probability = float(probabilities[a2_index, 1])  # 保存 A2 风险排序分数。
maximum_b_risk_probability = float(probabilities[b_component_test_indices, 1].max())  # 保存 B 分量最大风险分数。
a2_attention_rows = []  # 创建 A2 两层注意力账本。
for layer_index, attention in enumerate(final_attentions, start=1):  # 遍历两层 attention 概率。
    a2_attention_rows.append({"层": layer_index, "非零注意力": {nodes[index]["id"]: round(float(value), 4) for index, value in enumerate(attention[a2_index]) if float(value) > 1e-6}})  # 保存 A2 在每层实际关注节点。
print("测试节点逐样本结果：")  # 输出结果表标题。
pprint(result_rows)  # 展示默认 argmax 下的真实成功或失败。
print("A2 两层注意力：")  # 输出关键中间过程标题。
pprint(a2_attention_rows)  # 展示每层严格局部的概率分配。
print(f"预先固定决策=argmax/threshold {decision_threshold:.2f}；test accuracy：Baseline={baseline_test_accuracy:.3f}，GraphTransformer={graph_transformer_accuracy:.3f}")  # 明确规则和同测试集结果。
print(f"排序诊断：A2风险={a2_risk_probability:.4f}，B分量最高风险={maximum_b_risk_probability:.4f}，A2是否越过0.5={a2_risk_probability >= decision_threshold}")  # 区分排序能力与默认分类决策。
if graph_transformer_accuracy <= baseline_test_accuracy:  # 检查默认决策是否没有带来 accuracy 收益。
    print("诚实结论：模型把 A2 排在 B 分量之上，但默认 0.5 决策没有超过 baseline；需要独立验证集做校准，不能回看本测试集改阈值。")  # 明确报告受控实验局限。
else:  # 处理默认决策确实改善的情况。
    print("诚实结论：默认 argmax 在本固定测试集上改善 accuracy，但仍需独立图和时间切分验证。")  # 限定结果适用范围。

测试节点逐样本结果：
[{'Baseline': 0,
  'GraphTransformer@argmax': 0,
  '真实': 1,
  '节点': 'A2',
  '风险概率': 0.3658},
 {'Baseline': 0,
  'GraphTransformer@argmax': 0,
  '真实': 0,
  '节点': 'B3',
  '风险概率': 0.0001},
 {'Baseline': 0,
  'GraphTransformer@argmax': 0,
  '真实': 0,
  '节点': 'B4',
  '风险概率': 0.0001}]
A2 两层注意力：
[{'层': 1, '非零注意力': {'A1': 0.3333, 'A2': 0.3333, 'A3': 0.3333}},
 {'层': 2, '非零注意力': {'A1': 0.6232, 'A2': 0.1884, 'A3': 0.1884}}]
预先固定决策=argmax/threshold 0.50；test accuracy：Baseline=0.667，GraphTransformer=0.667
排序诊断：A2风险=0.3658，B分量最高风险=0.0001，A2是否越过0.5=False
诚实结论：模型把 A2 排在 B 分量之上，但默认 0.5 决策没有超过 baseline；需要独立验证集做校准，不能回看本测试集改阈值。


## 失败案例：取消图 mask 造成跨分量结构泄漏

如果把图当成全连接序列，B4 能在一层内直接关注完全不连通的 A0。即使概率不大，也违反业务图可达性；修正后的 mask 让这项概率精确为零。

In [6]:
full_mask = torch.ones_like(graph_mask, dtype=torch.bool)  # 构造错误的全连接可见性矩阵
with torch.no_grad():  # 关闭失败实验的梯度记录
    full_logits, full_hiddens, full_attentions, full_scores = model(features, full_mask)  # 用相同训练参数执行无图约束前向
b4_index = node_to_index["B4"]  # 查找不连通节点 B4 行号
a0_index = node_to_index["A0"]  # 查找欺诈种子 A0 列号
unsafe_cross_component_attention = float(full_attentions[0][b4_index, a0_index])  # 读取全连接时 B4 对 A0 的直接注意力
safe_cross_component_attention = float(final_attentions[0][b4_index, a0_index])  # 读取图 mask 下同一禁止位置概率
print("失败案例：B4 对不连通 A0 的第一层注意力", {"全连接": round(unsafe_cross_component_attention, 6), "图mask": safe_cross_component_attention})  # 展示结构泄漏及严格修正
print("B4 一跳合法可见节点：", [nodes[index]["id"] for index, allowed in enumerate(graph_mask[b4_index]) if bool(allowed)])  # 展示正确局部感受野

失败案例：B4 对不连通 A0 的第一层注意力 {'全连接': 0.175063, '图mask': 0.0}
B4 一跳合法可见节点： ['B3', 'B4']


## 生产差距

真实 Graph Transformer 还会使用多头 attention、边类型/最短路 bias、全局 token、稀疏 kernel 和大图采样。当前实验是 transductive：测试节点特征和边在训练时可见；若要声称 inductive，必须把新节点或整张新图完全隔离后再评测。风险阈值必须由独立验证集、预先声明的误报/漏报成本或业务容量选择，并锁定后只评一次测试集。图还必须按时间构建以防未来边泄漏；上线需监控邻居规模、截断损失、过平滑、校准、延迟和图更新新鲜度。

In [7]:
assert len(nodes) == 10 and len(edges) == 8  # 验证案例包含十个节点和两个不连通图分量。
assert training_ledger[-1]["loss"] < training_ledger[0]["loss"]  # 验证真实 backward 降低节点分类损失。
assert decision_threshold == 0.5 and torch.equal(predictions, final_logits.argmax(dim=1))  # 验证评估使用预先固定 argmax 而非测试集调阈值。
assert a2_risk_probability > maximum_b_risk_probability and 0.0 <= graph_transformer_accuracy <= 1.0  # 验证两跳节点排序证据并保留诚实分类结果。
assert unsafe_cross_component_attention > 0.0  # 验证全连接 attention 确实跨分量读取种子。
assert safe_cross_component_attention == 0.0  # 验证图 mask 把禁止位置概率严格归零。
print("最小回归测试通过：transductive 训练、固定 argmax、排序诊断与跨分量门禁均满足预期。")  # 输出集中断言的验收结论。

最小回归测试通过：transductive 训练、固定 argmax、排序诊断与跨分量门禁均满足预期。
